In [0]:
base_path = "/Volumes/data/raw/retail_db/"
file = "order_items"

In [0]:
schema_j = spark.read \
    .format("json") \
    .option("multiline", "true") \
    .load(f"{base_path}/schemas.json")

schema_j.display()

In [0]:
column_dict = schema_j.toPandas().to_dict(orient='list')

In [0]:
# file = "customers"
print((column_dict[file][0]))

In [0]:
cols = []

for column in (column_dict[file][0]):
    cols.append(f"{column['column_name']} {column['data_type']}")
    # print(column['column_name'], column['data_type'])

# print(cols)

schema = ",".join(cols)

print(schema)

In [0]:
# schema = "order_id integer,order_date string,order_customer_id string,order_status string"
# schema = "product_id integer,product_category_id integer,product_name string,product_description string,product_price float,product_image string"

In [0]:

df = spark.read \
    .format("csv") \
    .schema(schema) \
    .load(f"{base_path}/{file}")

df.printSchema()

In [0]:
# print(df.rdd.getNumPartitions())
# spark.conf.get('spark.sql.files.maxPartitionBytes')
# len(df.inputFiles())
# df.inputFiles()
spark.conf.get("spark.sql.adaptive.enabled")
# spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")
# df_products.cache()
# display(df)


In [0]:
# df.createOrReplaceTempView("orders_vw")

In [0]:
%sql
-- select count(*) from orders_vw where order_status in ('COMPLETE','CLOSED')

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 15)


In [0]:
df__count = df.groupBy("order_item_order_id").count().orderBy("order_item_order_id")

In [0]:
# print(df__count.rdd.getNumPartitions())
# df__count = df__count.coalesce(15)
# display(df__count)
display(df)

In [0]:
from pyspark.sql import functions as F

df__count.count()
# result = df_cat_count.agg(F.min("product_category_id"), F.max("product_category_id")).first()

In [0]:
# print(result)

In [0]:
from pyspark.sql.functions import col, to_date

df = df.withColumn("order_date", to_date(col("order_date")))

In [0]:
# df = df.repartition(64)
df = df.repartition(15)

In [0]:

partition_by = "json_data"
json_data_file = "json_data_file"

df.write \
    .format("json") \
    .mode("overwrite") \
    .partitionBy("order_item_product_id") \
    .save(f"/Volumes/data/raw/partitioned_data/retail_db/json_data_small_size_m/{file}")

# df.write \
#     .format("json") \
#     .mode("overwrite") \
#     .save(f"/Volumes/data/raw/partitioned_data/retail_db/json_data_file/{file}")


In [0]:
# dbutils.fs.rm("/Volumes/data/raw/partitioned_data/retail_db/orders/", recurse=True)
# dbutils.fs.ls("/Volumes/data/raw/partitioned_data/")
# dbutils.fs.ls("/Volumes/data/raw/partitioned_data/retail_db/")


In [0]:
# dbutils.fs.cp("/Volumes/data/raw/sales/partitioned_data/", "/Volumes/data/raw/partitioned_data/", recurse=True)

In [0]:
# dbutils.fs.ls("/Volumes/data/raw/partitioned_data/retail_db/")

In [0]:
# dbutils.fs.mkdirs("/Volumes/data/raw/dev/tpch")
# dbutils.fs.ls("/Volumes/data/raw/dev/")

In [0]:
df_orders = spark.read \
    .format("csv") \
    .option("header", True) \
    .load("/Volumes/data/raw/partitioned_data/retail_db/order_items")

In [0]:
df_orders.createOrReplaceTempView("order_items_vw")

In [0]:
%sql
select * from order_items_vw where order_item_order_id = 50023 and order_item_quantity > 1;
-- describe orders_vw;